# Score a 10x repertoire against peptides

Companion to https://tcrchallenge.org/tutorials/repertoire/. Read that guide for input definitions and interpretation. Run the cells in order in the same kernel.

Complete the setup guide first. Run from the DecoderTCR checkout pinned to `3e3d9889d26d79635f674940ddab039c4dc6f9f9`, with the 300M checkpoint downloaded. No model inference has been run in this notebook.

Download https://tcrchallenge.org/downloads/tutorial_helpers.py and place it beside this notebook. Read its documented pairing or PSSM conventions.

For repertoire analysis supply your own `filtered_contig_annotations.csv` and `peptide_targets.csv`. For design workflows inspect the public example and replace it with your chosen context. Record the model, source revision, device, files and settings. Generated sequences are candidates, not validated binders. Source and adapted code: MIT license.

## Choose the right input route

This notebook starts from raw Cell Ranger `filtered_contig_annotations.csv` plus a `peptide_targets.csv` table. It pairs chains with `prepare_10x` and scores the resulting receptor-target combinations. A processed `.h5ad` file is a different input route and must not be passed to this raw-contig helper.

The [10x worked example](https://tcrchallenge.org/tutorials/repertoire/#worked-example) starts with paired receptors, experimental binding labels and gene-expression data. It compares receptor scores for `GILGFVFTL` and `ELAGIGILTV`, each with `HLA-A*02:01`, against the assay labels and maps predictions back to cells. Use that example's input-preparation helper and documented label definitions for a processed dataset.


In [ ]:
import pandas as pd
from tutorial_helpers import prepare_10x

# One sample per call. Supply the HLA context in the target table.
contigs = pd.read_csv("filtered_contig_annotations.csv", keep_default_na=False)
targets = pd.read_csv("peptide_targets.csv", keep_default_na=False)
queries, audit = prepare_10x(
    contigs.to_dict("records"), targets.to_dict("records"), sample_id="sample_01"
)
pd.DataFrame(audit).to_csv("pairing_audit.csv", index=False)
if not queries:
    raise ValueError("No unambiguous paired receptors. Inspect pairing_audit.csv.")
query_table = pd.DataFrame(queries)
query_table.to_csv("repertoire_queries.csv", index=False)
print(f"{len(query_table)} receptor-target comparisons prepared.")

In [ ]:
import DecoderTCR as dt

# First inspect a small batch. To score the full table, set first_run=False.
first_run = True
selected = query_table.head(3) if first_run else query_table
scores = dt.score_from_components(
    selected, model="DecoderTCR-ESMC_300M", device="cpu"
)
metadata = selected[["name", "sample_id", "barcode", "target_index"]]
result = metadata.merge(scores, on="name", validate="one_to_one")
result.to_csv("repertoire_scores.csv", index=False)
print(result[["name", "ok", "tcr_reason", "hla_reason",
              "pll_DecoderTCR-ESMC_300M"]])

In [ ]:
import json
from pathlib import Path
run_record = {
    "source_revision": "3e3d9889d26d79635f674940ddab039c4dc6f9f9",
    "model": "DecoderTCR-ESMC_300M", "device": "cpu",
    "guide": "https://tcrchallenge.org/tutorials/repertoire/", "notebook": "repertoire",
    "note": "Save this notebook with the actual inputs and generation settings used."
}
Path("repertoire-run.json").write_text(json.dumps(run_record, indent=2))